# Results

Estimation results are stored in 3 files: the point estimates, the performance (RMSE), and out-of-sample estimates

In [ ]:
import numpy as np
import pandas as pd
import os
PATH_OUTPUT = f"{os.getcwd()}/../output"

date_str = "20260505"

forecast_file = f"fcst_{date_str}.xlsx"
performance_file = f"perf_{date_str}.xlsx"
outofsample_file = f"oos_{date_str}.xlsx"


results_table = pd.read_excel(f"{PATH_OUTPUT}/{forecast_file}", index_col='date')
results_table.index = pd.to_datetime(results_table.index, format="%Y-%m-%d")
performance_table = pd.read_excel(f"{PATH_OUTPUT}/{performance_file}", index_col='Vintage')
#outofsample_table = pd.read_excel(f"{PATH_OUTPUT}/{outofsample_file}", index_col='date')

In [ ]:
performance_table

In [ ]:
performance_table['spec'].unique()

In [ ]:
results_table

In [ ]:
results_table['spec'].unique()

# Presenting Results

In [ ]:
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
import seaborn as sns

# Define the color scheme
idb_blue = "#005BAC"
idb_light_blue = "#7FB3E6"

## Average RMSE by publication lag

In [ ]:
all_means = performance_table.groupby(performance_table.index)['RMSE'].mean()


fig, axes = plt.subplots(1, 1, figsize=(8, 5))  # adjust figsize as you want
# If axes is a single AxesSubplot (not an array), make it a list for consistency
axes = [axes]

# Plot
sns.lineplot(data=performance_table, x='Vintage', y='RMSE', ax=axes[0], color=idb_blue)

axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
axes[0].set_title("")
axes[0].tick_params(axis='x', rotation=45)
plt.tight_layout()

# Save figure:
plt.savefig(f'{PATH_OUTPUT}/Figure Vintage.png')
plt.show()

## Boxplot by publication lag

In [ ]:
df = performance_table.reset_index()

fig, ax = plt.subplots(figsize=(8, 5))

# If you want a specific order, define it here; otherwise seaborn will infer it.
# order = sorted(df['Publication lag'].unique())
order = None

sns.boxplot(
    data=df,
    x='Vintage',
    y='RMSE',
    order=order,
    ax=ax,
    boxprops=dict(color=idb_light_blue),
    whiskerprops=dict(color=idb_blue),
    capprops=dict(color=idb_blue),
    medianprops=dict(color='black'),
    flierprops=dict(markerfacecolor=idb_light_blue, markeredgecolor=idb_blue,
                    marker='o', markersize=6)
)

ax.set_title("")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df = performance_table.reset_index()

fig, ax = plt.subplots(figsize=(8, 5))

# If you want a specific order, define it here; otherwise seaborn will infer it.
# order = sorted(df['Publication lag'].unique())
order = None

sns.boxplot(
    data=df,
    x='Vintage',
    y='RMSE',
    order=order,
    ax=ax,
    boxprops=dict(color=idb_light_blue),
    whiskerprops=dict(color=idb_blue),
    capprops=dict(color=idb_blue),
    medianprops=dict(color='black'),
    flierprops=dict(markerfacecolor=idb_light_blue, markeredgecolor=idb_blue,
                    marker='o', markersize=6)
)

# Get the actual category order used on the axis (this is key)
cats = [t.get_text() for t in ax.get_xticklabels()]

# Means computed in the same category space as the plot
means = df.groupby('Vintage', dropna=False)['RMSE'].mean()

# Map tick labels -> x positions
xpos = np.arange(len(cats))

# Overlay means (match tick labels carefully)
ymeans = []
for c in cats:
    # Try direct label
    if c in means.index:
        ymeans.append(means.loc[c])
        continue
    # Try numeric label -> numeric index match (common when lag is int/float)
    try:
        cnum = float(c)
        hit = None
        for idx in means.index:
            try:
                if float(idx) == cnum:
                    hit = means.loc[idx]
                    break
            except Exception:
                pass
        ymeans.append(hit)
    except Exception:
        ymeans.append(None)

# Plot only the ones we found
mask = [v is not None and np.isfinite(v) for v in ymeans]
ax.scatter(
    xpos[mask],
    np.array(ymeans, dtype=float)[mask],
    color='black',
    marker='o',
    s=50,
    zorder=10,
    edgecolor='k',
    linewidth=0.4,
    label='Mean'
)

# Clean legend (only "Mean" once)
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys())

ax.set_title("")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## Histogram: distribution of errors

In [ ]:
human_mean = performance_table[performance_table['spec']=='human1']['RMSE'].mean()

plt.figure(figsize=(8, 6))

# Plot with Seaborn, no edge lines
sns.histplot(performance_table['RMSE'], bins=30, kde=False, color=idb_blue, edgecolor=None)

# Median line
plt.axvline(human_mean, color='black', linestyle=':', linewidth=2, label=f'Human mean = {human_mean:.2f}')

plt.title("")
plt.xlabel("Root Mean Squared Error (RMSE)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

**Filter out extreme outliers using the vintage mean**

In [ ]:
spec_rmse_mean = pd.DataFrame(performance_table.groupby('Vintage')['RMSE'].median())
spec_rmse_mean = spec_rmse_mean.rename(columns={'RMSE': 'threshold'})
spec_rmse_mean

In [ ]:
performance_table = performance_table.merge(spec_rmse_mean, on='Vintage', how='left')

# ++++++++++++++++++++++++++++++++++++++++++++++++++++
# ++++++++++++++++++++++++++++++++++++++++++++++++++++
filtered = performance_table[performance_table['RMSE'] < performance_table['threshold']]
# ++++++++++++++++++++++++++++++++++++++++++++++++++++
# ++++++++++++++++++++++++++++++++++++++++++++++++++++


human_mean = performance_table[performance_table['spec']=='human1']['RMSE'].mean()

plt.figure(figsize=(8, 6))

# Plot with Seaborn, no edge lines
sns.histplot(filtered['RMSE'], bins=30, kde=False, color=idb_blue, edgecolor=None)

# Median line
plt.axvline(human_mean, color='black', linestyle=':', linewidth=2, label=f'Human mean = {human_mean:.2f}')

plt.title("")
plt.xlabel("Root Mean Squared Error (RMSE)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

## Histogram conditional on publication lag

In [ ]:
human_means = (
    performance_table[performance_table['spec'] == 'human1']
    .groupby('Vintage')
    .mean(numeric_only=True)
)

# Mapping of vintages
lags = {
    -2: 'two_back',
    -1: 'one_back',
     0: 'zero_back',
     1: 'one_ahead',
     2: 'two_ahead'
}

fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 12), sharex=False, sharey=False)
axes = axes.flatten()

# ------------------------------------------------
# Loop over each vintage and plot histogram
for i, (V, N) in enumerate(lags.items()):
    ax = axes[i]

    # Filter data for this vintage
    subset = filtered[filtered.index == V]

    # Histogram
    sns.histplot(subset['RMSE'], bins=30, kde=False, color=idb_blue, edgecolor=None, ax=ax)

    # Median line
    rmse_median = subset['RMSE'].median()
    ax.axvline(rmse_median, color='black', linestyle='dashed', linewidth=2, label=f'Median = {rmse_median:.2f}')

    # Human benchmark
    human_mean = human_means[human_means.index==V]['RMSE'].mean()
    ax.axvline(human_mean, color='black', linestyle=':', linewidth=2, label=f'Human mean = {human_mean:.2f}')

    # Titles
    ax.set_title(f'Information set p={V}', fontsize=12)
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8, loc='upper right')

# ------------------------------------------------
# Last panel: pooled distribution
ax = axes[-1]
sns.histplot(filtered['RMSE'], bins=30, kde=False, color=idb_blue, edgecolor=None, ax=ax)

rmse_median = filtered['RMSE'].median()
ax.axvline(rmse_median, color='black', linestyle='dashed', linewidth=2, label=f'Median = {rmse_median:.2f}')

human_mean = performance_table[performance_table['spec'] == 'human1']['RMSE'].mean()
ax.axvline(human_mean, color='black', linestyle=':', linewidth=2, label=f'Human mean = {human_mean:.2f}')

ax.set_title('All Vintages', fontsize=12)
ax.set_ylabel('Frequency')
ax.legend(fontsize=8, loc='upper left')

# ------------------------------------------------
# X-labels on bottom row
for ax in axes[-3:]:
    ax.set_xlabel("Root Mean Squared Error (RMSE)")

plt.tight_layout()
plt.show()

## Boxplot (outliers management)

In [ ]:
# First we define reference points
human_p = performance_table[performance_table['spec']=='human1']
human_p = human_p.groupby(human_p.index)['RMSE'].mean()
all_means = filtered.groupby(filtered.index)['RMSE'].mean()


fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=filtered.reset_index(),
    x='Vintage',
    y='RMSE',
    ax=ax,
    boxprops=dict(color=idb_light_blue),
    whiskerprops=dict(color=idb_blue),
    capprops=dict(color=idb_blue),
    medianprops=dict(color='black'),
    flierprops=dict(markerfacecolor=idb_light_blue, markeredgecolor=idb_blue, marker='o', markersize=6)
)

# Ensure the figure is rendered so tick labels are available
fig.canvas.draw()   # <- important

xticks = ax.get_xticks()
xtlabels = [t.get_text() for t in ax.get_xticklabels()]

# compute means (keep original dtype)
all_means = filtered.reset_index().groupby('Vintage')['RMSE'].mean()

def find_mean_for_label(lbl):
    # direct match
    if lbl in all_means.index:
        return all_means.loc[lbl]
    # try numeric match
    try:
        lbl_num = float(lbl)
        for idx in all_means.index:
            try:
                if float(idx) == lbl_num:
                    return all_means.loc[idx]
            except Exception:
                continue
    except Exception:
        pass
    # fallback: match by string representation
    for idx in all_means.index:
        if str(idx) == lbl:
            return all_means.loc[idx]
    return None

# plot mean markers aligned to the xtick positions
for i, lbl in enumerate(xtlabels):
    m = find_mean_for_label(lbl)
    if m is not None:
        ax.scatter(xticks[i], m, color='black', marker='o', s=50, zorder=10,
                   label='Mean' if i == 0 else "", edgecolor='k', linewidth=0.4)

# overlay human points: match similarly
for i, vintage in enumerate(human_p.index):
    # try to find which xtick corresponds
    for j, lbl in enumerate(xtlabels):
        try:
            if str(vintage) == lbl or float(vintage) == float(lbl):
                ax.scatter(xticks[j], human_p[vintage], color='black', marker='v', s=50,
                           label='Human' if i == 0 else "")
                break
        except Exception:
            if str(vintage) == lbl:
                ax.scatter(xticks[j], human_p[vintage], color='black', marker='v', s=50,
                           label='Human' if i == 0 else "")
                break

# clean legend
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys())

ax.set_title("")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


# Nowcast outputs:

## Results/RMSE merge

Filter out results from under-performing models

In [ ]:
# -----------------------------------------------
# Reshape results to long format
df = results_table.reset_index().melt(
    id_vars=["date", 'spec', 'estimator'],
    value_vars=["two_back", "one_back", "zero_back", "one_ahead", "two_ahead"],
    var_name="horizon",
    value_name="value"
)

# Make Vintage column based on horizon -2, -1, 0, 1, 2
df['Vintage'] = df['horizon'].map({
    'two_back': -2,
    'one_back': -1,
    'zero_back': 0,
    'one_ahead': 1,
    'two_ahead': 2
})
df

In [ ]:
p = performance_table.reset_index()
results_w = (
    df  # safe even if index is unnamed
    .rename(columns=str.strip)
    .merge(
        p.rename(columns=str.strip)[['Vintage','spec','estimator','RMSE']],
        on=['Vintage','spec','estimator'],
        how='left'
    )
)
results_w

In [ ]:
mean_rmse = results_w.groupby('Vintage')['RMSE'].mean()

# Keep rows with RMSE <= Vintage-specific mean
filtered = results_w[
    results_w['RMSE'] <= results_w['Vintage'].map(mean_rmse)
]
filtered

## Ridge plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df = filtered[['date', 'horizon', 'Vintage', 'value']]
observed = pd.DataFrame(results_table['actuals'].groupby(results_table.index).mean())
actuals_dict = observed['actuals'].to_dict()
df

In [ ]:
# Mock data to make the code runnable
dates = df["date"].unique()
horizons = ["two_back", "one_back", "zero_back", "one_ahead", "two_ahead"]
data = []
for date in dates:
    for horizon in horizons:
        # Create different distribution shapes to make the plot more interesting
        values = df[(df['date']==date) & (df['horizon']==horizon)].value
        data.extend([(date, horizon, v) for v in values])

df = pd.DataFrame(data, columns=["date", "horizon", "value"])

horizon_labels = {
    "two_back": "p=-2",
    "one_back": "p=-1",
    "zero_back": "p=0",
    "one_ahead": "p=1",
    "two_ahead": "p=2"
}
df["horizon"] = df["horizon"].map(horizon_labels)

sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

# Generate a range of colors for the horizons
horizons_unique = df["horizon"].unique()
n_horizons = len(horizons_unique)
pal = sns.cubehelix_palette(n_horizons, rot=-.25, light=.7)

# The fix is to add sharey=False
g = sns.FacetGrid(
    df,  
    row="date",  
    hue="horizon",  
    aspect=5,  
    height=2,  
    palette=pal,
    sharey=True  # This is the key change
)

# KDE distributions
g.map(
    sns.kdeplot,  
    "value",  
    bw_adjust=0.7,  
    clip_on=False,  
    fill=True,  
    alpha=0.6,  
    linewidth=1.2,
    common_norm=True
)

# Contour lines
g.map(
    sns.kdeplot,  
    "value",  
    bw_adjust=0.7,  
    clip_on=False,  
    color="w",  
    lw=0.5
)

# Adjust spacing between ridges
g.fig.subplots_adjust(hspace=-0.2)

# Y-axis labels as dates, remove density scale
for ax, date in zip(g.axes.flat, df["date"].unique()):
    ax.set_ylabel(date.strftime("%Y-%m-%d"), rotation=0, labelpad=50, ha="right", va="center", fontsize=18)
    ax.set_yticks([])
    ax.axhline(y=0, color="gray", lw=0.6, ls="--", alpha=0.7)

# Add observed (example without results_table)
# Loop through the axes and sorted dates to plot the correct 'actual' value
sorted_dates = sorted(df["date"].unique())
for ax, date in zip(g.axes.flat, sorted_dates):
    actual_value = actuals_dict.get(date)
    if actual_value is not None:
        # Create a line with y-coordinates from 0.1 to 0.9 of the axes height,
        # but with the x-coordinate in data units.
        line = plt.Line2D(
            [actual_value, actual_value], # x-coordinates in data units
            [0, 0.9],                   # y-coordinates in axes units
            transform=ax.transData,       # This is the correct transform object
            color='black', linestyle='-', lw=1, alpha=0.8
        )
        # Add the line to the current axes.
        ax.add_line(line)
     
# Cleanup axes
g.set(xlabel="GDP growth QoQ%")
g.set_titles("")
g.despine(bottom=True, left=True)

# Custom white legend
palette_dict = dict(zip(horizons_unique, pal))
handles = [mpatches.Patch(color=palette_dict[h], label=h) for h in horizons_unique]
plt.legend(
    handles=handles,  
    title="Information",  
    title_fontsize=18,  
    fontsize=18,  
    bbox_to_anchor=(1.2, 4),  
    loc="upper center",
    frameon=True,
    facecolor="white",
    edgecolor="gray",
    framealpha=1
)

# Save and show
plt.show()

## Weighted Nowcast

We will weight the nowcast by the inverse of the RMSE. Effectively, we will compute:

$$
\hat{v}
= \frac{\sum_{i=1}^{N} w_i \, v_i}{\sum_{i=1}^{N} w_i}
= \frac{\sum_{i=1}^{N} \dfrac{v_i}{\mathrm{RMSE}_i}}
       {\sum_{i=1}^{N} \dfrac{1}{\mathrm{RMSE}_i}}
$$

We can also estimate a weighted confidence interval using the same weights.

$$
\mathrm{SE}(\hat{v})
= \frac{1}{\sqrt{\sum_{i=1}^{N} w_i}}
= \left( \sum_{i=1}^{N} \frac{1}{\mathrm{RMSE}_i} \right)^{-1/2}
$$

$$
\hat{v} \pm 1.96 \cdot \mathrm{SE}(\hat{v})
$$



In [ ]:
filtered = filtered.copy()
filtered['date'] = pd.to_datetime(filtered['date'])
filtered['weight'] = 1 / filtered['RMSE']
filtered

In [ ]:
df = results_table[['actuals']]
df = df.groupby('date').mean()
df.index = pd.to_datetime(df.index)
df

In [ ]:
# ---------------------------------------------------
V = -2
N = "two_back"

d = filtered.loc[filtered['Vintage'] == V]
weighted = (
    (d['value'] * d['weight']).groupby(d['date']).sum()
    / d.groupby(d['date'])['weight'].sum()
).reset_index(name=N).set_index('date')


df = df.merge(weighted, left_index=True, right_index=True, how='left')
df

In [ ]:
# ---------------------------------------------------
V = -1
N = "one_back"

d = filtered.loc[filtered['Vintage'] == V]
weighted = (
    (d['value'] * d['weight']).groupby(d['date']).sum()
    / d.groupby(d['date'])['weight'].sum()
).reset_index(name=N).set_index('date')

df = df.merge(weighted, left_index=True, right_index=True, how='left')

# ---------------------------------------------------
V = 0
N = "zero_back"

d = filtered.loc[filtered['Vintage'] == V]
weighted = (
    (d['value'] * d['weight']).groupby(d['date']).sum()
    / d.groupby(d['date'])['weight'].sum()
).reset_index(name=N).set_index('date')

df = df.merge(weighted, left_index=True, right_index=True, how='left')

# ---------------------------------------------------
V = 1
N = "one_ahead"

d = filtered.loc[filtered['Vintage'] == V]
weighted = (
    (d['value'] * d['weight']).groupby(d['date']).sum()
    / d.groupby(d['date'])['weight'].sum()
).reset_index(name=N).set_index('date')

df = df.merge(weighted, left_index=True, right_index=True, how='left')

# ---------------------------------------------------
V = 2
N = "two_ahead"

d = filtered.loc[filtered['Vintage'] == V]
weighted = (
    (d['value'] * d['weight']).groupby(d['date']).sum()
    / d.groupby(d['date'])['weight'].sum()
).reset_index(name=N).set_index('date')

df = df.merge(weighted, left_index=True, right_index=True, how='left')

df

Alternatively... a loop, from start:

In [ ]:
horizons = {
    'two_back': -2,
    'one_back': -1,
    'zero_back': 0,
    'one_ahead': 1,
    'two_ahead': 2
}

filtered = filtered.copy()
filtered['date'] = pd.to_datetime(filtered['date'])
filtered['weight'] = 1 / filtered['RMSE']

df = results_table[['actuals']]
df = df.groupby('date').mean()
df.index = pd.to_datetime(df.index)
df

In [ ]:
horizons = {
    'two_back': -2,
    'one_back': -1,
    'zero_back': 0,
    'one_ahead': 1,
    'two_ahead': 2
}

for N, V in horizons.items():
    d = filtered.loc[filtered['Vintage'] == V]

    weighted = (
        (d['value'] * d['weight']).groupby(d['date']).sum()
        / d.groupby(d['date'])['weight'].sum()
    ).reset_index(name=N).set_index('date')

    df = df.merge(weighted, left_index=True, right_index=True, how='left')

df


## Graph results

In [ ]:
plot_df = df.rename(columns={
    'actuals': 'observed',
    'two_back': 'p=-2',
    'one_back': 'p=-1',
    'zero_back': 'p=0',
    'one_ahead': 'p=1',
    'two_ahead': 'p=2'
})
plot_df.index = pd.PeriodIndex(plot_df.index, freq='Q').astype(str)
fig, ax = plt.subplots(figsize=(10,6))

# Plot 'observed' as a black dashed line, no marker
plot_df['observed'].plot(
    ax=ax, color='black', linestyle='-', linewidth=2, label='observed'
)

# Define line styles and markers for other lines
line_styles = ['--', '--', '-.', ':', (0, (3, 5, 1, 5))]  # various dash styles
markers = ['x', 'o', '^', 'v', 'D']  # circle, square, triangle up/down, diamond

# Plot the rest with unique line styles and markers
for idx, col in enumerate(['p=-2', 'p=-1', 'p=0', 'p=1', 'p=2']):
    plot_df[col].plot(
        ax=ax,
        color=idb_blue,
        linestyle=line_styles[idx],
        marker=markers[idx],
        linewidth=1.5,
        markersize=5,
        label=col
    )

# Labels and grid
ax.set_title('')
ax.set_xlabel('Quarter', fontsize=12)
ax.set_ylabel('QoQ GDP Growth (%)', fontsize=12)
ax.legend(title='Publication Lag', fontsize=12, title_fontsize=12)
plt.tight_layout()

plt.show()

## Weighted Nowcast with Confidence Intervals

In [ ]:
df = results_table[['actuals']].groupby('date').mean()
df

In [ ]:
lags = {
    -2: 'two_back',
    -1: 'one_back',
     0: 'zero_back',
     1: 'one_ahead',
     2: 'two_ahead'
}

for V, N in lags.items():
    # Read and prepare results_table fresh in each loop
    New_Table = filtered.copy()
    d = New_Table.loc[New_Table['Vintage'] == V]

    g = d.groupby('date')

    meanN = (
        (d['value'] * d['weight']).groupby(d['date']).sum()
        / g['weight'].sum()
    ).rename(N)

    seN = (1 / np.sqrt(g['weight'].sum())).rename(f'se_{N}')

    weighted = pd.concat([meanN, seN], axis=1)

    # Compute 95% confidence interval bounds
    weighted[f'ci_lower_{N}'] = weighted[N] - 1.96 * weighted[f'se_{N}']
    weighted[f'ci_upper_{N}'] = weighted[N] + 1.96 * weighted[f'se_{N}']

    # Concatenate the new columns to the main df
    df = df.join(weighted[[N, f'se_{N}', f'ci_lower_{N}', f'ci_upper_{N}']])

df.index = pd.PeriodIndex(df.index, freq='Q').astype(str)
df

In [ ]:
lags = {
    -2: 'two_back',
    -1: 'one_back',
     0: 'zero_back',
     1: 'one_ahead',
     2: 'two_ahead'
}

fig, axes = plt.subplots(nrows=5, ncols=1, figsize=(10, 20), sharex=True)

# ------------------------------------------------
# Loop over each vintage and plot
for ax, (V, N) in zip(axes, lags.items()):
    # Plot actuals
    ax.plot(df.index, df['actuals'], color='black', linestyle='-', label='Observed')
    
    # Plot nowcast for vintage N
    ax.plot(df.index, df[N], color=idb_blue, linestyle='--', marker='x', label=f'Weighted Nowcast')
    
    # Plot confidence interval as a shaded area
    ax.fill_between(
        df.index,
        df[f'ci_lower_{N}'],
        df[f'ci_upper_{N}'],
        color=idb_light_blue,
        alpha=0.4,
        label='95% CI'
    )
    
    # Titles and labels
    ax.set_title(f'Information set p={V}', fontsize=12)
    ax.set_ylabel('QoQ GDP Growth (%)')
    ax.legend(fontsize=9, loc='upper right')

axes[-1].set_xlabel('Quarter')

plt.tight_layout()
plt.show()

# Out of sample

In [ ]:
import os
import pandas as pd
import numpy as np
try :
    date_str
except :
    date_str = "20260505"


PATH_OUTPUT = os.path.join(os.getcwd(), "..", "output/", f"{date_str}")
PATH_DATA = os.path.join(os.getcwd(), "..", "data/")


import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
import seaborn as sns
'''
# Define the color scheme
idb_blue = "#005BAC"
idb_light_blue = "#7FB3E6"

# Load results
forecast_file = f"{date_str} 3_nwcst Tab fcst.xlsx"
performance_file = f"{date_str} 3_nwcst Tab perf.xlsx"
outofsample_file = f"{date_str} 3_nwcst Tab oos.xlsx"


results_table = pd.read_excel(f"{PATH_OUTPUT}/{forecast_file}", index_col='date', parse_dates=True)
results_table.index = pd.to_datetime(results_table.index, format="%Y-%m-%d")
performance_table = pd.read_excel(f"{PATH_OUTPUT}/{performance_file}", parse_dates=True)
'''

forecast = filtered.copy()
forecast = forecast.dropna(how='all', axis=0)

performance = performance_table.copy()
performance = performance.dropna(how='any', axis=0)
performance['inv_rmse'] = 1 / (performance['RMSE']** 2)
performance['weight'] = performance.groupby('Vintage')['inv_rmse'].transform(lambda x: x / x.sum())

performance.reset_index(drop=False, inplace=True)


performance = performance[['Vintage', 'estimator', 'spec', 'weight']]
performance = performance.set_index('Vintage')
performance.index = performance.index.astype(str).str.replace("-2",'two_back')
performance.index = performance.index.astype(str).str.replace("-1",'one_back')
performance.index = performance.index.astype(str).str.replace("0",'zero_back')
performance.index = performance.index.astype(str).str.replace("1",'one_ahead')
performance.index = performance.index.astype(str).str.replace("2",'two_ahead')
performance = performance.reset_index()

performance.rename(columns={'Vintage': 'horizon'}, inplace=True)

#forecast = forecast.reset_index().melt(id_vars=['date', 'actuals', 'estimator', 'spec'],
#        value_vars=['two_back', 'one_back', 'zero_back', 'one_ahead', 'two_ahead'],
#        var_name='Vintage',
#        value_name='value')
forecast = forecast.merge(performance, on=['horizon', 'estimator', 'spec'])
forecast = forecast.dropna(how='all', axis=0)
forecast = forecast[~forecast['value'].isna()]
forecast['value'] = ( ((1+ ( forecast['value']/100) )**4)-1 )*100
performance

In [ ]:
data = pd.read_csv(f"{PATH_DATA}/data.csv", parse_dates=["date"], index_col='date')
data = data[['RGDP0000']].dropna()

from statsmodels.tsa.seasonal import seasonal_decompose
data['RGDP0000'] = seasonal_decompose(data['RGDP0000'].dropna(how='all', axis=0), model='additive', extrapolate_trend='freq', period=4).trend

'''
data = pd.read_csv(f"{PATH_FORMAT}/data_full.csv", parse_dates=["date"], index_col='date')['RGDP0000'].dropna()
'''
gdp = data.resample('Q').agg({'RGDP0000': 'mean'})
gdp.index = gdp.index.to_period('Q').asfreq('M').to_timestamp()
gdp = (gdp.pct_change(periods=1)).dropna()*100
gdp.rename(columns = { 'RGDP0000' : 'actuals' } , inplace=True)
gdp['actuals'] = (((1+gdp['actuals']/100)**4) - 1)*100

actuals = pd.DataFrame(gdp['actuals'].groupby(gdp.index).mean())

avg = pd.DataFrame(forecast.groupby('date')['value'].mean()).rename(columns = {'value' : 'avg'})

est_fcst = pd.DataFrame( forecast.groupby(['date', 'estimator'])['value'].mean() ).rename(columns = {'value' : 'avg'}).reset_index()
gbt_fcst = est_fcst[est_fcst['estimator']=='GBT'].set_index('date')[['avg']].rename(columns = {'avg' : 'GBT'})

avg = pd.concat([avg, gbt_fcst], axis=1)

wavg = forecast.groupby('date').apply(lambda g: np.average(g['value'], weights=g['weight']) if g['weight'].sum() > 0 else np.nan).reset_index(name='weighted_avg').set_index('date')

avg = pd.concat([avg, wavg], axis=1)
avg.index = pd.to_datetime(avg.index)
avg

In [ ]:
out_of_sample = pd.read_excel(f"{PATH_OUTPUT}/{outofsample_file}", parse_dates=True)
out_of_sample['date'] = pd.to_datetime(out_of_sample['date'])
out_of_sample['nowcast'] = (((1+out_of_sample['nowcast']/100)**4) - 1)*100


# Filter out of sample by RMSE
filtered_grouped = (
    filtered[filtered['Vintage'] == 2]
    .groupby(['spec', 'estimator'], as_index=False)
    .agg({
        'RMSE': 'mean',
        'threshold': 'mean'
    })
)

out_of_sample = out_of_sample.merge(
    filtered_grouped,
    on=['spec', 'estimator'],
    how='left'
)
out_of_sample.dropna(subset=['RMSE'], inplace=True)
out_of_sample = out_of_sample[out_of_sample['RMSE'] <= out_of_sample['threshold']]
out_of_sample

In [ ]:

#lower_bound = out_of_sample['nowcast'].quantile(0.10)
#upper_bound = out_of_sample['nowcast'].quantile(0.90)
#out_of_sample['nowcast'] = out_of_sample['nowcast'].where((out_of_sample['nowcast'] >= lower_bound) & (out_of_sample['nowcast'] <= upper_bound), np.nan)
'''
quantiles = out_of_sample.groupby('date')['nowcast'].quantile([0.10, 0.90]).unstack()
out_of_sample = out_of_sample.merge(quantiles, on='date', suffixes=('', '_quantile'))
out_of_sample['nowcast'] = out_of_sample['nowcast'].where(
    (out_of_sample['nowcast'] >= out_of_sample[0.10]) & 
    (out_of_sample['nowcast'] <= out_of_sample[0.90]), 
    np.nan
)
out_of_sample = out_of_sample.drop(columns=[0.10, 0.90])
'''

out_of_sample = out_of_sample.merge(pd.DataFrame(performance.groupby(['estimator', 'spec'])['weight'].mean()).reset_index(), on=['estimator', 'spec']).set_index('date')

oos_avg = pd.DataFrame(out_of_sample.groupby(out_of_sample.index)['nowcast'].mean()).rename(columns = {'nowcast' : 'avg'})

oos_gbt = pd.DataFrame(out_of_sample.groupby(['date', 'estimator'])['nowcast'].mean().reset_index()).set_index('date').rename(columns = {'nowcast' : 'GBT'})
oos_gbt = pd.DataFrame(oos_gbt[oos_gbt['estimator'] == 'GBT']['GBT'])
oos_avg = pd.concat([oos_avg, oos_gbt], axis=1)

out_of_sample = out_of_sample.dropna(how='any', axis=0)
wavg = out_of_sample.groupby(out_of_sample.index).apply(lambda g: np.average(g['nowcast'], weights=g['weight']) if g['weight'].sum() > 0 else np.nan).reset_index(name='weighted_avg').set_index('date')
oos_avg = pd.concat([oos_avg, wavg], axis=1)

out_of_sample = out_of_sample.dropna(how='any', axis=0)
dfrange = out_of_sample.groupby(out_of_sample.index)['nowcast'].agg(min='min', max='max').reset_index().set_index('date')
df_ci = (
    out_of_sample
    .groupby(out_of_sample.index)['nowcast']
    .quantile([0.05, 0.95])
    .unstack()
    .rename(columns={0.05: 'ci_lower', 0.95: 'ci_upper'})
)
oos_avg = pd.concat([oos_avg, dfrange, df_ci], axis=1)

oos_avg.index = pd.to_datetime(oos_avg.index)
oos_avg

In [ ]:
data = pd.concat( [ avg , oos_avg ] , axis=1 )
data = data.groupby(level=0, axis=1).sum(min_count=1)
data

In [ ]:
data = pd.concat([ data, actuals ] , axis=1)
data = pd.concat([ data, gdp ] , axis=1)
data

In [ ]:
import matplotlib.pyplot as plt

dataframe = data.copy()

# Ensure the last observed point has no interval
last_valid_idx = dataframe['actuals'].last_valid_index()
dataframe.at[last_valid_idx, 'ci_lower'] = dataframe.at[last_valid_idx, 'weighted_avg']
dataframe.at[last_valid_idx, 'ci_upper'] = dataframe.at[last_valid_idx, 'weighted_avg']

# Restrict sample
dataframe = dataframe[dataframe.index > "2022-01-01"]

plt.figure(figsize=(10, 6))

# Confidence interval band
plt.fill_between(
    dataframe.index,
    dataframe['ci_lower'],
    dataframe['ci_upper'],
    color='grey',
    alpha=0.15,
    edgecolor='none',
    label="_nolegend_"
)

# Observed GDP
plt.plot(
    dataframe.index,
    dataframe['actuals'],
    color='black',
    marker='>',
    linewidth=2,
    label='Observed GDP'
)

# Average nowcast
plt.plot(
    dataframe.index,
    dataframe['avg'],
    color='crimson',
    linestyle=":",
    marker='x',
    markersize=5,
    label='Average Nowcast'
)

# Weighted nowcast
plt.plot(
    dataframe.index,
    dataframe['weighted_avg'],
    color='indigo',
    linestyle="--",
    marker='x',
    markersize=8,
    label='RMSE-weighted Nowcast'
)

# Zero line
plt.axhline(y=0, color='black', linestyle='--', linewidth=1)

# Labels
plt.xlabel('Date', fontsize=12)
plt.ylabel('Real GDP Annualized Growth Rate (QoQ%)', fontsize=12)

# Rotate x labels
plt.xticks(rotation=45)

# Remove duplicate legend entries (safety)
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), loc='lower left')

# Layout and save
plt.tight_layout()
plt.savefig(f"{PATH_OUTPUT}/{date_str} 4_rslt Fig out_of_sample.png",
            dpi=300,
            bbox_inches='tight')

plt.show()